# 04 — Community Detection

**Purpose:** Partition the build dependency graph into communities to
identify cohesive groups of targets and potential merge candidates.  
We apply Louvain and Leiden algorithms, characterise each community, find
bridging targets, and rank merge candidates by coupling/cohesion.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_optimiser.graph import load_graph, attach_metrics
from build_optimiser.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")

# Load processed data
file_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "file_metrics.parquet")
target_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "target_metrics.parquet")
edge_list = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "edge_list.parquet")

# Load dependency graph
G = load_graph(str(PROJECT_ROOT / "data" / "raw" / "dot"))
attach_metrics(G, target_metrics)

print(f"Files: {len(file_metrics)}, Targets: {len(target_metrics)}, Edges: {len(edge_list)}")

In [ ]:
# ── Louvain community detection with resolution sweep ────────────────
from networkx.algorithms.community import louvain_communities

# Convert directed graph to undirected for community detection
G_undirected = G.to_undirected()

resolutions = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
louvain_results = {}

for res in resolutions:
    communities = louvain_communities(G_undirected, resolution=res, seed=42)
    louvain_results[res] = communities
    modularity = nx.algorithms.community.modularity(G_undirected, communities)
    print(f"Resolution {res:.2f}: {len(communities)} communities, modularity={modularity:.4f}")

# Plot number of communities vs resolution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

n_comms = [len(louvain_results[r]) for r in resolutions]
mods = [nx.algorithms.community.modularity(G_undirected, louvain_results[r]) for r in resolutions]

ax1.plot(resolutions, n_comms, "o-", color="steelblue")
ax1.set_xlabel("Resolution")
ax1.set_ylabel("Number of communities")
ax1.set_title("Louvain: communities vs resolution")

ax2.plot(resolutions, mods, "s-", color="coral")
ax2.set_xlabel("Resolution")
ax2.set_ylabel("Modularity")
ax2.set_title("Louvain: modularity vs resolution")

plt.tight_layout()
plt.show()

# Select best resolution (highest modularity)
best_res = resolutions[np.argmax(mods)]
best_communities = louvain_results[best_res]
print(f"\nSelected resolution: {best_res} ({len(best_communities)} communities, modularity={max(mods):.4f})")

In [ ]:
# ── Leiden community detection for comparison ────────────────────────
# Leiden requires the `leidenalg` package.  Fall back gracefully.

try:
    import leidenalg
    import igraph as ig

    # Convert networkx graph to igraph
    node_list = list(G_undirected.nodes())
    node_idx = {n: i for i, n in enumerate(node_list)}
    ig_edges = [(node_idx[u], node_idx[v]) for u, v in G_undirected.edges()]
    ig_graph = ig.Graph(n=len(node_list), edges=ig_edges, directed=False)
    ig_graph.vs["name"] = node_list

    leiden_partition = leidenalg.find_partition(ig_graph, leidenalg.ModularityVertexPartition, seed=42)
    leiden_communities = [set(ig_graph.vs[idx]["name"] for idx in part) for part in leiden_partition]
    leiden_modularity = leiden_partition.modularity

    print(f"Leiden: {len(leiden_communities)} communities, modularity={leiden_modularity:.4f}")

    # Build node-to-community mapping for Leiden
    leiden_map = {}
    for cid, comm in enumerate(leiden_communities):
        for node in comm:
            leiden_map[node] = cid

    LEIDEN_AVAILABLE = True

except ImportError:
    print("leidenalg not installed — skipping Leiden detection.")
    print("Install with: pip install leidenalg")
    LEIDEN_AVAILABLE = False
    leiden_communities = best_communities  # fallback to Louvain
    leiden_map = {}

In [ ]:
# ── Community characterisation ───────────────────────────────────────
# For each Louvain community: size, total compile time, internal edges
# (cohesion), and external edges (coupling).

# Build node-to-community mapping
node_to_comm = {}
for cid, comm in enumerate(best_communities):
    for node in comm:
        node_to_comm[node] = cid

records = []
for cid, comm in enumerate(best_communities):
    sub = G.subgraph(comm)
    internal_edges = sub.number_of_edges()

    # External edges: edges crossing community boundary
    external_edges = 0
    for node in comm:
        for nbr in G.successors(node):
            if nbr not in comm:
                external_edges += 1
        for nbr in G.predecessors(node):
            if nbr not in comm:
                external_edges += 1

    total_ct = sum(G.nodes[n].get("compile_time_s", 0) or 0 for n in comm)

    cohesion = internal_edges / max(len(comm) * (len(comm) - 1) / 2, 1)
    coupling = external_edges / max(internal_edges + external_edges, 1)

    records.append({
        "community": cid,
        "size": len(comm),
        "total_compile_time_s": total_ct,
        "internal_edges": internal_edges,
        "external_edges": external_edges,
        "cohesion": cohesion,
        "coupling": coupling,
    })

comm_df = pd.DataFrame(records).sort_values("total_compile_time_s", ascending=False).reset_index(drop=True)
print(comm_df.to_string(index=False))

# Summary plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(range(len(comm_df)), comm_df["size"], color="steelblue", edgecolor="black")
axes[0].set_xlabel("Community")
axes[0].set_ylabel("Size (# targets)")
axes[0].set_title("Community sizes")

axes[1].bar(range(len(comm_df)), comm_df["total_compile_time_s"], color="coral", edgecolor="black")
axes[1].set_xlabel("Community")
axes[1].set_ylabel("Total compile time (s)")
axes[1].set_title("Compile time by community")

axes[2].scatter(comm_df["cohesion"], comm_df["coupling"], s=comm_df["size"] * 3,
                alpha=0.7, edgecolors="black")
axes[2].set_xlabel("Cohesion (internal density)")
axes[2].set_ylabel("Coupling (external edge ratio)")
axes[2].set_title("Cohesion vs Coupling")
for _, row in comm_df.iterrows():
    axes[2].annotate(str(int(row["community"])), (row["cohesion"], row["coupling"]), fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ── Bridging targets via betweenness centrality ──────────────────────
# Targets with high betweenness centrality act as bridges between
# communities.  Splitting or optimising them can decouple the graph.

print("Computing betweenness centrality (this may take a moment)...")
bc = nx.betweenness_centrality(G, normalized=True)
bc_df = pd.DataFrame({"target": list(bc.keys()), "betweenness": list(bc.values())})
bc_df["community"] = bc_df["target"].map(node_to_comm)
bc_df = bc_df.sort_values("betweenness", ascending=False).reset_index(drop=True)

print("\nTop-20 bridging targets (highest betweenness centrality):")
print(bc_df.head(20).to_string(index=False))

# Plot
top_bc = bc_df.head(20)
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(top_bc)), top_bc["betweenness"].values, color="mediumpurple", edgecolor="black")
ax.set_yticks(range(len(top_bc)))
ax.set_yticklabels([str(t)[:30] for t in top_bc["target"].values], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Betweenness centrality")
ax.set_title("Top-20 bridging targets")
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualisation (force-directed layout coloured by community) ──────
# For large graphs, sample a manageable subset.

MAX_VIS_NODES = 300
if G.number_of_nodes() > MAX_VIS_NODES:
    # Keep the top communities by size, then sample within each
    sampled_nodes = []
    for cid, comm in enumerate(best_communities):
        comm_list = list(comm)
        n_sample = max(1, int(MAX_VIS_NODES * len(comm) / G.number_of_nodes()))
        sampled_nodes.extend(np.random.RandomState(42).choice(comm_list, size=min(n_sample, len(comm_list)), replace=False))
    vis_sub = G.subgraph(sampled_nodes).copy()
else:
    vis_sub = G.copy()

# Assign community colours
n_communities = len(best_communities)
cmap = plt.cm.get_cmap("tab20", max(n_communities, 1))
node_colors = [cmap(node_to_comm.get(n, 0) % 20) for n in vis_sub.nodes()]

fig, ax = plt.subplots(figsize=(16, 12))
pos = nx.spring_layout(vis_sub, k=1.5 / np.sqrt(max(vis_sub.number_of_nodes(), 1)), seed=42, iterations=60)

nx.draw_networkx_nodes(vis_sub, pos, node_color=node_colors, node_size=30, alpha=0.8, ax=ax)
nx.draw_networkx_edges(vis_sub, pos, alpha=0.15, arrows=True, arrowsize=5, width=0.3, ax=ax)

ax.set_title(f"Build graph coloured by Louvain community (resolution={best_res}, {n_communities} communities)",
             fontsize=13)
ax.axis("off")

# Legend for top communities
legend_n = min(10, n_communities)
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=cmap(i), label=f"Community {i} (n={len(best_communities[i])})")
                   for i in range(legend_n)]
ax.legend(handles=legend_elements, loc="upper left", fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Merge candidates ranking ─────────────────────────────────────────
# Pairs of communities with high inter-community edge density relative
# to their sizes are good merge candidates (they are tightly coupled).

from itertools import combinations

pair_records = []
for ci, cj in combinations(range(len(best_communities)), 2):
    comm_i = best_communities[ci]
    comm_j = best_communities[cj]

    # Count cross edges
    cross = 0
    for u, v in G.edges():
        if (u in comm_i and v in comm_j) or (u in comm_j and v in comm_i):
            cross += 1

    if cross == 0:
        continue

    # Normalise by the maximum possible cross edges
    max_cross = len(comm_i) * len(comm_j)
    density = cross / max_cross if max_cross > 0 else 0

    merged_size = len(comm_i) + len(comm_j)
    merged_ct = (sum(G.nodes[n].get("compile_time_s", 0) or 0 for n in comm_i) +
                 sum(G.nodes[n].get("compile_time_s", 0) or 0 for n in comm_j))

    pair_records.append({
        "comm_a": ci,
        "comm_b": cj,
        "size_a": len(comm_i),
        "size_b": len(comm_j),
        "cross_edges": cross,
        "cross_density": density,
        "merged_size": merged_size,
        "merged_compile_time_s": merged_ct,
    })

merge_df = pd.DataFrame(pair_records).sort_values("cross_density", ascending=False).reset_index(drop=True)

print("Top-15 merge candidates (highest cross-edge density):")
print(merge_df.head(15).to_string(index=False))

# Heatmap of cross-edge density
n_c = len(best_communities)
density_matrix = np.zeros((n_c, n_c))
for _, row in merge_df.iterrows():
    a, b = int(row["comm_a"]), int(row["comm_b"])
    density_matrix[a, b] = row["cross_density"]
    density_matrix[b, a] = row["cross_density"]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(density_matrix, cmap="YlOrRd", ax=ax, xticklabels=range(n_c), yticklabels=range(n_c))
ax.set_title("Cross-community edge density")
ax.set_xlabel("Community")
ax.set_ylabel("Community")
plt.tight_layout()
plt.show()